# Day 26: Zero-Downtime Deployment & Cost Estimation
> *100 Days of Inference* | Layer: **Infrastructure** | Book: *Inference Engineering* Ch 7.4 (pp. 200–202)

**Prerequisite:** Day 25

## What problem does this solve?

Inference Application Programming Interfaces (APIs) cannot take downtime for model swaps. New checkpoints, quantization changes, runtime upgrades, configuration tweaks — all of it has to ship while live traffic continues. The deployment strategy determines how much capacity overhead the swap costs and how visible the swap is to end users. Once the swap is solved, the next operational question is what all the hardware actually costs per million tokens served.

## Concept Overview

Three deployment strategies bracket the safety/capacity tradeoff:

- **Blue-green:** Two identical fleets. Blue serves; green is the new version. Once green passes health checks, the load balancer flips weights atomically. Rollback is a single weight flip. Cost: 2× Graphics Processing Unit (GPU) capacity during the overlap.
- **Rolling update:** Replace replicas one at a time. Cheap (no extra capacity), but exposes partial failures because some traffic always hits the in-flight version.
- **Canary:** Shift traffic in steps (1% → 5% → 25% → 100%) with automatic rollback on threshold breach. Slowest path, but catches subtle regressions that only appear at small traffic percentages.

Cost estimation reduces to one formula:

$$\frac{\$}{\text{1M tokens}} = \frac{\text{GPU \$/hr}}{\text{tokens/sec} \times \text{utilization} \times 3600} \times 10^6$$

Three levers: GPU rate, throughput (set by High-Bandwidth Memory bandwidth and quantization), and utilization. Idle GPUs cost the same as busy ones — utilization is the silent cost killer.

In [1]:
!pip install -q numpy matplotlib torch 2>/dev/null
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA GB10
GPU Memory: 128.5 GB


## Part 1: Blue-Green Deployment

Maintain two identical fleets. Deploy the new version to the idle fleet, validate it against health checks and a hold-out traffic slice, then atomically switch traffic via the load balancer. Rollback is one weight flip.

Tradeoff: 2× GPU cost during the overlap window (typically 15–30 minutes for a careful cutover).

In [2]:
class BlueGreenDeployment:
    def __init__(self):
        self.blue_weight = 1.0
        self.green_weight = 0.0
        self.state = "blue_only"

    def deploy_green(self, green_version: str):
        self.state = "deploying_green"
        print(f"[DEPLOY] Starting green fleet with {green_version}...")
        return self

    def validate_green(self, health_checks_passed: bool, error_rate: float) -> bool:
        if health_checks_passed and error_rate < 0.01:
            self.state = "green_validated"
            print(f"[OK] Green validated: health={health_checks_passed}, err={error_rate:.2%}")
            return True
        print(f"[FAIL] Green validation failed: err={error_rate:.2%}")
        return False

    def shift_traffic(self, target_green_pct: float):
        self.green_weight = target_green_pct
        self.blue_weight = 1.0 - target_green_pct
        print(f"[TRAFFIC] Blue: {self.blue_weight:.0%}, Green: {self.green_weight:.0%}")

    def complete_rollout(self):
        self.green_weight = 1.0; self.blue_weight = 0.0; self.state = "green_only"
        print("[COMPLETE] Rollout complete. Green is now primary.")

    def rollback(self):
        self.green_weight = 0.0; self.blue_weight = 1.0; self.state = "blue_only"
        print("[ROLLBACK] Rolled back to blue fleet.")

deployment = BlueGreenDeployment()
deployment.deploy_green("Llama-3.1-8B-Instruct v2")
time.sleep(0.05)

if deployment.validate_green(health_checks_passed=True, error_rate=0.003):
    deployment.shift_traffic(0.10)
    time.sleep(0.05)
    deployment.shift_traffic(0.50)
    time.sleep(0.05)
    deployment.complete_rollout()

print(f"Final state: {deployment.state}")

[DEPLOY] Starting green fleet with Llama-3.1-8B-Instruct v2...
[OK] Green validated: health=True, err=0.30%
[TRAFFIC] Blue: 90%, Green: 10%
[TRAFFIC] Blue: 50%, Green: 50%
[COMPLETE] Rollout complete. Green is now primary.
Final state: green_only


## Part 2: Canary Deployment with Auto-Rollback

Shift traffic incrementally (1% → 5% → 10% → 25% → 50% → 100%). At each step, validate that error rate and p99 latency stay within thresholds (typical gate: 2× baseline error, 1.5× baseline p99). Auto-rollback fires on any breach.

Slower than blue-green (30–60 minutes for full rollout) but catches regressions that only show up at small traffic slices — things like prompt-format edge cases or new-quant numerical drift.

In [3]:
class CanaryDeployment:
    def __init__(self, steps=None,
                 baseline_error_rate=0.005,
                 baseline_p99_ms=500.0):
        self.steps = steps or [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
        self.current_step = 0
        self.canary_pct = 0.0
        self.baseline_error_rate = baseline_error_rate
        self.baseline_p99_ms = baseline_p99_ms
        self.history = []

    def _gate(self, err: float, p99_ms: float) -> bool:
        return err < self.baseline_error_rate * 2 and p99_ms < self.baseline_p99_ms * 1.5

    def advance(self, metrics: dict) -> bool:
        ok = self._gate(metrics["error_rate"], metrics["p99_latency_ms"])
        self.history.append({**metrics, "canary_pct": self.canary_pct, "passed": ok})
        if not ok:
            print(f"  [ROLLBACK] Gate failed at {self.canary_pct:.0%}: "
                  f"err={metrics['error_rate']:.3%}, p99={metrics['p99_latency_ms']:.0f}ms")
            self.canary_pct = 0.0
            return False
        if self.current_step < len(self.steps):
            self.canary_pct = self.steps[self.current_step]
            self.current_step += 1
            print(f"  [ADVANCE] Canary at {self.canary_pct:.0%}: "
                  f"err={metrics['error_rate']:.3%}, p99={metrics['p99_latency_ms']:.0f}ms")
        return True

# Scenario A: smooth rollout
print("Scenario A: healthy canary\n" + "-" * 40)
np.random.seed(5)
canary = CanaryDeployment()
for _ in range(6):
    canary.advance({"error_rate": 0.003 + np.random.normal(0, 0.001),
                    "p99_latency_ms": 350 + np.random.normal(0, 20)})
print(f"Final canary pct: {canary.canary_pct:.0%}\n")

# Scenario B: latency regression at 10%
print("Scenario B: regression triggers rollback\n" + "-" * 40)
np.random.seed(11)
canary = CanaryDeployment()
p99_per_step = [380, 420, 820, 0, 0, 0]  # spike at step 3
for p99 in p99_per_step:
    if p99 == 0: break
    canary.advance({"error_rate": 0.004,
                    "p99_latency_ms": p99 + np.random.normal(0, 10)})
print(f"Final canary pct after rollback: {canary.canary_pct:.0%}")

Scenario A: healthy canary
----------------------------------------
  [ADVANCE] Canary at 1%: err=0.344%, p99=343ms
  [ADVANCE] Canary at 5%: err=0.543%, p99=345ms
  [ADVANCE] Canary at 10%: err=0.311%, p99=382ms
  [ADVANCE] Canary at 25%: err=0.209%, p99=338ms
  [ADVANCE] Canary at 50%: err=0.319%, p99=343ms
  [ADVANCE] Canary at 100%: err=0.181%, p99=346ms
Final canary pct: 100%

Scenario B: regression triggers rollback
----------------------------------------
  [ADVANCE] Canary at 1%: err=0.400%, p99=397ms
  [ADVANCE] Canary at 5%: err=0.400%, p99=417ms
  [ROLLBACK] Gate failed at 5%: err=0.400%, p99=815ms
Final canary pct after rollback: 0%


## Part 3: Cost Per Million Tokens

The cost formula breaks into three levers:

- **GPU rate:** H100 ≈ $3.50–4.50/hr on-demand, B200 ≈ $6/hr, A100 ≈ $2.50/hr.
- **Throughput per GPU:** Set by High-Bandwidth Memory (HBM) bandwidth and quantization. INT4 reads ¼ the weight bytes per token versus FP16, so throughput scales accordingly.
- **Utilization:** The silent killer. Idle GPUs cost the same as busy ones — a fleet running at 30% utilization costs ~2.3× more per token than the same fleet at 70%.

Comparing configurations and quantizations across models reveals where the cost is actually spent.

In [4]:
def cost_per_million_tokens(gpu_cost_per_hr, n_gpus, tokens_per_sec_per_gpu, utilization=0.7):
    """$/1M tokens at given utilization."""
    cost_per_sec = gpu_cost_per_hr * n_gpus / 3600
    actual_tps = tokens_per_sec_per_gpu * n_gpus * utilization
    return cost_per_sec / actual_tps * 1e6

configs = [
    ("7B   on 1xH100  (FP16)",  3.5, 1, 1500),
    ("7B   on 1xH100  (INT4)",  3.5, 1, 2500),
    ("70B  on 4xH100  (FP16)",  3.5, 4,  300),
    ("70B  on 4xH100  (FP8) ",  3.5, 4,  600),
    ("70B  on 4xH100  (INT4)",  3.5, 4,  800),
    ("671B on 8xH100  (FP8) ",  3.5, 8,   80),
    ("7B   on 1xA100  (FP16)",  2.5, 1, 1200),
]

print("Cost per 1M tokens at 70% utilization")
print(f"{'Config':<26} {'Total $/hr':>11} {'TPS':>6}  {'$/1M tok':>10}")
print("-" * 60)
for name, hr, n, tps in configs:
    cpm = cost_per_million_tokens(hr, n, tps, 0.7)
    print(f"{name:<26} ${hr*n:>9.2f} {tps:>6} ${cpm:>9.3f}")

print("\nUtilization sensitivity (70B on 4xH100, FP16, 300 TPS)")
print(f"{'Utilization':>12} {'$/1M tokens':>14}")
print("-" * 30)
for u in [0.2, 0.3, 0.5, 0.7, 0.9]:
    print(f"{u:>11.0%}  ${cost_per_million_tokens(3.5, 4, 300, u):>12.3f}")

print("\nQuantization is a 4-8x cost lever; utilization is a 4x lever.")
print("Both compound: INT4 at 70% util costs ~12x less than FP16 at 30% util.")

# Capacity planning: given target traffic, how many servers and what's the bill?
def capacity_plan(monthly_tokens, gpu_cost_per_hr, n_gpus_per_server,
                  tokens_per_sec_per_server, target_utilization=0.7):
    avg_tps = monthly_tokens / (30 * 24 * 3600)
    required_tps = avg_tps / target_utilization
    n_servers = int(np.ceil(required_tps / tokens_per_sec_per_server))
    monthly_cost = n_servers * n_gpus_per_server * gpu_cost_per_hr * 24 * 30
    return n_servers, monthly_cost, monthly_cost / (monthly_tokens / 1e6)

print("\nCapacity planning: 70B FP8 on 4xH100 ($14/hr/server, 2400 TPS/server, 70% util)")
for tokens in [1e9, 10e9, 100e9]:
    n_srv, total, per_m = capacity_plan(tokens, 3.5, 4, 600 * 4, 0.7)
    print(f"  {tokens/1e9:>5.0f}B tokens/mo -> {n_srv:>3} servers, "
          f"${total:>11,.0f}/mo, ${per_m:.3f}/1M tok")

Cost per 1M tokens at 70% utilization
Config                      Total $/hr    TPS    $/1M tok
------------------------------------------------------------
7B   on 1xH100  (FP16)     $     3.50   1500 $    0.926
7B   on 1xH100  (INT4)     $     3.50   2500 $    0.556
70B  on 4xH100  (FP16)     $    14.00    300 $    4.630
70B  on 4xH100  (FP8)      $    14.00    600 $    2.315
70B  on 4xH100  (INT4)     $    14.00    800 $    1.736
671B on 8xH100  (FP8)      $    28.00     80 $   17.361
7B   on 1xA100  (FP16)     $     2.50   1200 $    0.827

Utilization sensitivity (70B on 4xH100, FP16, 300 TPS)
 Utilization    $/1M tokens
------------------------------
        20%  $      16.204
        30%  $      10.802
        50%  $       6.481
        70%  $       4.630
        90%  $       3.601

Quantization is a 4-8x cost lever; utilization is a 4x lever.
Both compound: INT4 at 70% util costs ~12x less than FP16 at 30% util.

Capacity planning: 70B FP8 on 4xH100 ($14/hr/server, 2400 TPS/serv

## Part 4: Graceful Shutdown and Request Draining

When a Kubernetes pod is being replaced, three things must happen in order:

1. **preStop hook fires** — pod is removed from the load balancer endpoints, no new requests arrive.
2. **Drain phase** — in-flight requests complete, up to `terminationGracePeriodSeconds`.
3. **SIGTERM** — process exits cleanly. (After the grace period, SIGKILL is unconditional.)

Drain timeout sizing: set it to p99 request duration plus buffer. Too short kills long-tail requests mid-stream and produces user-visible errors during every rollout. Too long and rollouts crawl, blocking subsequent deploys.

In [5]:
class GracefulShutdown:
    def __init__(self, drain_timeout_s: float = 300.0):
        self.drain_timeout = drain_timeout_s
        self.accepting_new = True
        self.active_requests: list[int] = []
        self.start_time: float | None = None

    def begin_drain(self, active_request_count: int):
        self.accepting_new = False
        self.start_time = time.time()
        self.active_requests = list(range(active_request_count))
        print(f"[DRAIN] Started with {active_request_count} active requests, "
              f"timeout {self.drain_timeout:.0f}s.")

    def complete_request(self, req_id: int):
        if req_id in self.active_requests:
            self.active_requests.remove(req_id)

    def status(self) -> str:
        elapsed = time.time() - self.start_time if self.start_time else 0
        remaining = len(self.active_requests)
        if remaining == 0:
            return f"DRAINED in {elapsed:.1f}s"
        if elapsed > self.drain_timeout:
            return f"TIMEOUT: {remaining} requests force-terminated"
        return f"DRAINING: {remaining} active, {elapsed:.1f}s elapsed"

shutdown = GracefulShutdown(drain_timeout_s=60.0)
shutdown.begin_drain(active_request_count=25)
for req_id in range(25):
    shutdown.complete_request(req_id)
print(shutdown.status())
print("\nKubernetes lifecycle:")
print("  preStop hook -> deregister from LB -> drain -> SIGTERM -> SIGKILL (after grace)")

[DRAIN] Started with 25 active requests, timeout 60s.
DRAINED in 0.0s

Kubernetes lifecycle:
  preStop hook -> deregister from LB -> drain -> SIGTERM -> SIGKILL (after grace)


## Try These Experiments

1. **Latency-gated canary:** Extend `CanaryDeployment` to use p99 Time-To-First-Token (TTFT) instead of total p99 latency. Trigger rollback if TTFT exceeds 1.5× baseline. Plot the traffic split and the rollback decision over a 60-minute simulated rollout.
2. **Break-even utilization:** For a 70B model on 4× H100 at $3.50/hr each, what utilization is needed to match a managed Application Programming Interface (API) priced at $1 per million tokens? Assume 300 tokens/sec at full utilization. Solve symbolically, then verify with `cost_per_million_tokens`.
3. **Rolling update headroom:** During a rolling update with N replicas, one is offline at a time. What is the maximum traffic level (as a fraction of normal capacity) the cluster can absorb without queueing? Compute for N ∈ {2, 4, 8, 16}.

## Key Takeaways

- Blue-green = instant rollback, 2× GPU cost during the overlap window. Best for high-risk swaps.
- Rolling update = cheapest, but exposes partial failures because both versions serve concurrently during the rollout.
- Canary = safest for catching quality regressions. Step the traffic up with auto-rollback gates.
- Cost formula: $/1M tokens = (GPU $/hr) ÷ (tokens/sec × utilization × 3600) × 10⁶. Quantization is a 4–8× lever; utilization is a 4× lever; the two compound.
- Idle GPUs cost the same as busy ones — autoscaling Return On Investment (ROI) tracks traffic variance directly.
- Drain timeout = p99 request duration + buffer. Smaller breaks long-tail requests; larger blocks subsequent rollouts.
- **What's next:** Day 27 — Performance benchmarking: tooling and profiling.

## References
- *Inference Engineering* Ch 7.4 (pp. 200–202)
- [Google SRE Book — Release Engineering](https://sre.google/sre-book/release-engineering/)